# Pricing American Options

Whereas a European option can be exercised only at a fixed date, an American option can be exercised any time up to its expiration, thus, the value of an American option is the value achieved by exercising optimally.
The problem is to solve the following optimization problem:
$ sup_{\tau \in \Tau} \mathbb{E}[U(\tau)]$, where $\tau$ representa a stopping time taking value in $[0,T]$ and $U$ represents the discounted payoff from exercise at time t.

We begin by pricing an American Put with interest rate $r$ and divident yield $q$ constant and positive s.t. r > q, depending on only one asset which follows the dynamics of Black-Scholes model, using the Longstaff–Schwartz method, a standard approach in the literature.

Therefore, consider a version of Black-Scholes model in which the underlying asset has dividend yield $q$, then the risk-neutral dynamics of the asset is given by 
$$
\frac{dS_t}{S_t} = ( r - q ) dt + \sigma dW_t
$$

More specifically in the present simulation we are considering a Bermudan option where every discretized time point is a possible exercise date.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# Utility function for later use

# Simulation of the underlying asset price using Geometric Brownian Motion (GBM)
def simulate_gbm(S0, mu, sigma, T, dt):
	N = int(T / dt)  # Number of time steps
	t = np.linspace(0, T, N+1)  # Time vector
	W = np.random.standard_normal(size=N)  # Brownian motion increments
	W = np.insert(np.cumsum(W) * np.sqrt(dt), 0, 0) # Brownian motion path
	S = S0 * np.exp((mu - 0.5 * sigma**2) * t + sigma * W)
	return S

# More general regression basis functions
def regression_basis_fit(X, Y, basis_functions):
    # Each column is psi_j(x) where psi_j is the j-th basis function 
    Psi = np.column_stack([f(X) for f in basis_functions])
    # Solving the least-squared problem  Psi^T@ Psi beta ≈ Psi^T @Y
    #beta = np.linalg.solve(Psi.T @ Psi, Psi.T @ Y)
    beta = np.linalg.lstsq(Psi, Y, rcond=None)[0]
    return beta 

def regression_basis_predict(X, coeffs, basis_functions):
    Psi = np.column_stack([f(X) for f in basis_functions])
    return Psi @ coeffs

# Example for further use
basis_functions = [ 
	lambda x: np.ones_like(x),  
	lambda x: x,                
	lambda x: x**2              
]


In [3]:
# Parameters
S0 = 248  # Current price 
K = 280 # Strike price
sigma = 0.3 # Volatility 
r = 0.035  # Risk-free rate
q = 0.0043 # Dividend yield
mu = r - q  # Drift 
T = 1.0  # Maturity 
dt = 0.004  # Time step (daily 1/252)
N = int(T / dt)  # Number of steps 
b = 10000 # Number of simulated paths

# 1) Simulate N paths of the underlying asset price using GBM

Z = np.random.standard_normal((b, N)) # Increments
BM = np.hstack([np.zeros((b,1)), np.cumsum(Z * np.sqrt(dt), axis=1)]) 
time = np.arange(0, N + 1) * dt
drift = (mu - 0.5 * sigma**2) * time
paths = S0 * np.exp(drift + sigma * BM)

# 2) Calculate the payoff of the option at maturity for each path 
# This corresponds to the value of the option at maturity
value = np.zeros((b, N+1))
value[:, -1] = np.maximum(0, K - paths[:, -1] )  # Put option payoff at maturity

# 3) Backward induction to estimate the option price at time 0

for t in range(N - 1, -1, -1):
    # Discount the option values back to time t
    discounted_value = value[:, t + 1] * np.exp(-r * dt)
    # Use regression to estimate the continuation value
    X = paths[:, t]  # Stock price at time t
    Y = discounted_value  # Discounted option value at time t+1
    #LS algorithm does the regression only on the in-the-money paths
    itm = (K - X) > 0
    X_itm = X[itm] 
    Y_itm = discounted_value[itm] 
    # Regression 
    coefficients = np.polyfit(X_itm, Y_itm, deg=3)
    continuation_value = np.polyval(coefficients, X)
    # Determine the option value at time t
    payoff = np.maximum(0, K - X)  # Immediate exercise payoff
    value[:, t] = np.where( payoff > continuation_value, payoff, discounted_value)  

# 4) The option price at time 0 is the average of the discounted payoffs
option_price = np.mean(value[:, 0])
print(f"Estimated option price at time 0: {option_price:.2f}")

Estimated option price at time 0: 40.66


/var/folders/2h/wg2tcsjx2dv4v71hht7r1js00000gn/T/ipykernel_16781/2072952989.py:39: RankWarning: Polyfit may be poorly conditioned
  coefficients = np.polyfit(X_itm, Y_itm, deg=3)
